# Clase 193 — Stack bayesiano moderno: PyMC v5 + NumPyro + ArviZ

PyMC v5 (PyTensor backend), NumPyro (JAX, NUTS rápido), ArviZ (diagnósticos y comparación de modelos).
Requiere: `pip install numpy scipy arviz` (opcional `pymc`, `numpyro`).

In [ ]:
import numpy as np
rng = np.random.default_rng(42)
n = 200
TRUE_A, TRUE_B, TRUE_SIGMA = 1.5, 2.3, 1.0
x = rng.normal(0, 1, n)
y = TRUE_A + TRUE_B * x + rng.normal(0, TRUE_SIGMA, n)
print(f'n={n}  truth: a={TRUE_A}  b={TRUE_B}  sigma={TRUE_SIGMA}')

## Modelo bayesiano
$$ a \sim N(0,10),\;\; b \sim N(0,10),\;\; \sigma \sim \text{HalfNormal}(5),\;\; y_i \sim N(a + b x_i, \sigma) $$

## Opción A — PyMC v5

In [ ]:
PYMC_OK = False
try:
    import pymc as pm
    import arviz as az
    with pm.Model() as model:
        a = pm.Normal('a', 0, 10)
        b = pm.Normal('b', 0, 10)
        sigma = pm.HalfNormal('sigma', 5)
        mu = a + b * x
        pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y)
        idata = pm.sample(1000, tune=1000, chains=2, random_seed=42, progressbar=False)
    PYMC_OK = True
    print(az.summary(idata, var_names=['a','b','sigma'], round_to=3))
except Exception as e:
    print(f'PyMC no disponible ({type(e).__name__}); fallback al Metropolis manual abajo.')

## ArviZ — diagnósticos y plots

In [ ]:
if PYMC_OK:
    import matplotlib.pyplot as plt
    az.plot_trace(idata, var_names=['a','b','sigma']); plt.tight_layout(); plt.show()
    az.plot_posterior(idata, var_names=['a','b','sigma'], hdi_prob=0.94); plt.show()
    # rhat / ess
    rhat = az.rhat(idata)
    ess = az.ess(idata)
    print('rhat:', {k: float(v) for k,v in rhat.items() if k in ['a','b','sigma']})
    print('ess :', {k: float(v) for k,v in ess.items()  if k in ['a','b','sigma']})
    print('Regla: rhat < 1.01 y ess > 400/chain -> OK')
else:
    print('Skip ArviZ plots (PyMC no disponible).')

## Fallback — Metropolis-Hastings manual
Si PyMC no está, esto da una posterior cruda para verificar la lógica.

In [ ]:
from scipy.stats import norm, halfnorm

def log_post(a, b, sigma, x, y):
    if sigma <= 0: return -np.inf
    lp  = norm.logpdf(a, 0, 10) + norm.logpdf(b, 0, 10) + halfnorm.logpdf(sigma, scale=5)
    lp += norm.logpdf(y, loc=a + b*x, scale=sigma).sum()
    return lp

def metropolis(x, y, n_iter=8000, burn=2000, seed=42):
    r = np.random.default_rng(seed)
    a, b, s = 0.0, 0.0, 1.0
    chain = np.empty((n_iter, 3))
    cur = log_post(a, b, s, x, y)
    for i in range(n_iter):
        ap, bp, sp = a + r.normal(0, 0.15), b + r.normal(0, 0.15), s + r.normal(0, 0.1)
        prop = log_post(ap, bp, sp, x, y)
        if np.log(r.uniform()) < prop - cur:
            a, b, s, cur = ap, bp, sp, prop
        chain[i] = a, b, s
    return chain[burn:]

ch = metropolis(x, y)
print('Metropolis manual:')
print(f'  a    : mean={ch[:,0].mean():.3f}  HDI94=[{np.percentile(ch[:,0],3):.3f}, {np.percentile(ch[:,0],97):.3f}]')
print(f'  b    : mean={ch[:,1].mean():.3f}  HDI94=[{np.percentile(ch[:,1],3):.3f}, {np.percentile(ch[:,1],97):.3f}]')
print(f'  sigma: mean={ch[:,2].mean():.3f}  HDI94=[{np.percentile(ch[:,2],3):.3f}, {np.percentile(ch[:,2],97):.3f}]')

## Opción B — NumPyro (JAX, NUTS rápido)

In [ ]:
try:
    import numpyro
    import numpyro.distributions as dist
    from numpyro.infer import MCMC, NUTS
    import jax.numpy as jnp
    from jax import random as jrandom

    def model_np(x, y=None):
        a = numpyro.sample('a', dist.Normal(0, 10))
        b = numpyro.sample('b', dist.Normal(0, 10))
        sigma = numpyro.sample('sigma', dist.HalfNormal(5))
        numpyro.sample('y_obs', dist.Normal(a + b*x, sigma), obs=y)

    kernel = NUTS(model_np)
    mcmc = MCMC(kernel, num_warmup=500, num_samples=1000, num_chains=2, progress_bar=False)
    mcmc.run(jrandom.PRNGKey(42), x=jnp.array(x), y=jnp.array(y))
    mcmc.print_summary()
except ImportError:
    print('numpyro no instalado; `pip install numpyro` para NUTS sobre JAX.')

## Comparación de modelos con ArviZ (WAIC / LOO)
Si tenemos un modelo M1 (lineal) y M2 (cuadrático), ¿cuál predice mejor out-of-sample?

In [ ]:
if PYMC_OK:
    # Modelo cuadratico
    with pm.Model() as model2:
        a = pm.Normal('a', 0, 10)
        b = pm.Normal('b', 0, 10)
        c = pm.Normal('c', 0, 10)
        sigma = pm.HalfNormal('sigma', 5)
        mu = a + b*x + c*x**2
        pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y)
        idata2 = pm.sample(1000, tune=1000, chains=2, random_seed=42, progressbar=False, idata_kwargs={'log_likelihood': True})
    # Re-sample del modelo 1 con log_lik
    with model:
        pm.compute_log_likelihood(idata)
    cmp = az.compare({'linear': idata, 'cuadratico': idata2}, ic='loo')
    print(cmp)
    print('\nMejor = el de arriba (loo mas alto = mejor predictivo).')
else:
    print('Skip comparación (PyMC no disponible).')

## Diagnósticos clave
- **R-hat < 1.01**: cadenas convergieron.
- **ESS > 400/cadena**: suficiente información independiente.
- **Divergences = 0** (NUTS): si hay, reparametrizar o subir `target_accept`.
- **WAIC/LOO**: comparar predictivo entre modelos.

## Takeaways
1. **PyMC v5** = Python idiomatic, backend PyTensor, NUTS por default.
2. **NumPyro** = JAX → orden de magnitud más rápido en CPU/GPU, API similar.
3. **ArviZ** = lingua franca de inferencia bayesiana (idata, plots, comparación).
4. Workflow: definir modelo → sample → revisar rhat/ess/divergences → posterior plots → comparar con LOO.
5. Si nada de eso está disponible, Metropolis manual hace el trabajo conceptual — sólo no escalá.